# Лабораторна робота №2
# Думич Софія ОІ-35

**Тема:** Очищення даних та інженерія ознак
**Клієнт:** AstroVision Observatory  
### Варіант 1(8 порядковий у журналі)

Star Classification Dataset - Stellar Properties

**Опис Компанії-Замовника:** AstroVision Observatory - наукова організація, яка
займається автоматизованим аналізом астрономічних спостережень і
формуванням каталогів небесних об’єктів.

**Запит:** Обсерваторія отримує велику кількість записів про фізичні
характеристики зір. Ручне визначення типу кожного об’єкта потребує значного
часу, тому необхідно автоматизувати первинну класифікацію зір.

**Пропозиція Реалізації:** Розробити модель багатокласової класифікації, яка
визначатиме тип зорі на основі температури, світності, радіуса, маси, зоряної
величини, спектрального класу, кольору та інших характеристик. Модель
класифікуватиме об’єкти як коричневі карлики, червоні карлики, зорі головної
послідовності, білі карлики, гіганти, надгіганти або гіпергіганти.


## Імпорт бібліотек та завантаження даних

In [ ]:
%pip install scikit-learn

In [2]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

DB_PATH = "ЛАБ1/astro_vision.db"
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("SELECT * FROM stars", conn)
df.head()

,Star_ID,Temperature_K,Luminosity_L_Sun,Radius_R_Sun,Mass_M_Sun,Distance_LY,Apparent_Magnitude,Absolute_Magnitude,Spectral_Class,Star_Color,Age_Million_Years,Metallicity,Rotation_Speed_KM_S,Constellation,Galaxy_Type,Variable_Star,Surface_Gravity,Star_Type,timestamp
0,STAR_00001,4941.457,161.574,1.110,2.419,70.259,5.975,-0.691,K,Orange,1969.922,0.060,2.540,Lyra,Spiral,No,537.951,Giant,2025-12-23 12:40:22
1,STAR_00002,8323.197,1.057,0.653,0.411,27145.213,24.372,4.770,A,White,1040.618,0.040,10.098,Scorpius,Spiral,No,264.222,Main Sequence,2025-05-16 23:10:38
2,STAR_00003,4950.156,19.438,1.935,2.993,48346.438,22.463,1.608,K,Orange,2653.410,-0.019,6.565,Scorpius,Spiral,No,218.939,Giant,2025-11-22 00:07:56
3,STAR_00004,21960.502,0.007,1.099,1.014,11.347,12.906,10.198,B,Blue-White,11993.647,0.223,11.333,Scorpius,Spiral,No,230.090,White Dwarf,2025-11-27 16:27:49
4,STAR_00005,3375.070,0.031,0.550,0.088,1437.434,21.840,8.619,M,Red,7938.604,-0.375,1.469,Gemini,Spiral,No,79.636,Red Dwarf,2025-12-01 13:38:54


## Штучне створення дефектів даних

In [3]:
np.random.seed(42)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

if 'Star_ID' in cat_cols:
    cat_cols.remove('Star_ID')
if 'timestamp' in cat_cols:
    cat_cols.remove('timestamp')

for col in ['Temperature_K', 'Luminosity_L_Sun']:
    nan_indices = np.random.choice(df.index, size=int(len(df) * 0.05), replace=False)
    df.loc[nan_indices, col] = np.nan

for col in ['Star_Color', 'Spectral_Class']:
    nan_indices = np.random.choice(df.index, size=int(len(df) * 0.05), replace=False)
    df.loc[nan_indices, col] = np.nan

outlier_indices = np.random.choice(df.index, size=50, replace=False)
df.loc[outlier_indices, 'Temperature_K'] = df['Temperature_K'].max() * 5

df[df["Temperature_K"].isna() | df["Spectral_Class"].isna()].head(10)


,Star_ID,Temperature_K,Luminosity_L_Sun,Radius_R_Sun,Mass_M_Sun,Distance_LY,Apparent_Magnitude,Absolute_Magnitude,Spectral_Class,Star_Color,Age_Million_Years,Metallicity,Rotation_Speed_KM_S,Constellation,Galaxy_Type,Variable_Star,Surface_Gravity,Star_Type,timestamp
3,STAR_00004,NaN,0.007,1.099,1.014,11.347,12.906,10.198,B,Blue-White,11993.647,0.223,11.333,Scorpius,Spiral,No,230.090,White Dwarf,2025-11-27 16:27:49
14,STAR_00015,NaN,1.407,2.259,1.210,2474.304,18.860,4.459,A,White,3628.237,-0.362,7.546,Cassiopeia,Spiral,No,64.925,Main Sequence,2025-04-08 17:57:02
28,STAR_00029,3669.495,0.016,0.867,0.442,1189.481,22.136,9.326,NaN,Red,4314.491,-0.244,8.785,Scorpius,Spiral,No,161.335,Red Dwarf,2025-11-25 10:55:55
29,STAR_00030,NaN,0.046,1.229,0.953,249.620,17.583,8.163,A,White,11469.720,-0.189,13.458,Gemini,Spiral,No,173.077,White Dwarf,2025-02-04 15:33:34
31,STAR_00032,NaN,37476.840,15.840,59.343,890.221,5.576,-6.604,B,Blue-White,49.266,0.172,3.756,Aquila,Spiral,No,64.806,Supergiant,2025-08-02 11:28:45
33,STAR_00034,NaN,1.055,1.369,4.876,8118.134,21.752,4.772,F,Yellow-White,5264.631,-0.141,1.379,Gemini,Spiral,Yes,713.262,Main Sequence,2025-05-14 01:32:26
35,STAR_00036,NaN,0.035,0.379,0.563,3449.464,23.588,8.466,M,Red,9287.099,-0.430,10.740,Orion,Spiral,No,1074.024,Red Dwarf,2025-10-16 20:42:22
39,STAR_00040,NaN,3.871,1.783,3.924,44.357,9.028,3.360,G,Yellow,5222.202,0.062,20.435,Orion,Spiral,No,338.058,Main Sequence,2025-06-30 18:18:12
50,STAR_00051,3277.160,0.015,0.977,0.546,6.517,10.888,9.385,NaN,Red,9569.382,-0.116,6.534,Pegasus,Spiral,Yes,156.815,Red Dwarf,2025-06-02 14:06:57
60,STAR_00061,6344.092,6.720,2.366,3.452,212.424,11.830,2.762,NaN,Yellow-White,2621.758,-0.143,3.941,Virgo,Spiral,No,168.916,Main Sequence,2025-12-29 16:22:06


## Перевірка дефектів у даних

In [4]:
print("Числові:", num_cols)
print("Категоріальні:", cat_cols)
print("\nПропуски:")
print(df.isna().sum()[lambda s: s > 0].to_string())
print("\nПриклад «брудних» значень Spectral_Class:", df["Spectral_Class"].dropna().unique()[:12])

Числові: ['Temperature_K', 'Luminosity_L_Sun', 'Radius_R_Sun', 'Mass_M_Sun', 'Distance_LY', 'Apparent_Magnitude', 'Absolute_Magnitude', 'Age_Million_Years', 'Metallicity', 'Rotation_Speed_KM_S', 'Surface_Gravity']
Категоріальні: ['Spectral_Class', 'Star_Color', 'Constellation', 'Galaxy_Type', 'Variable_Star', 'Star_Type']

Пропуски:
Temperature_K       499
Luminosity_L_Sun    500
Spectral_Class      500
Star_Color          500

Приклад «брудних» значень Spectral_Class: ['K' 'A' 'B' 'M' 'O' 'G' 'F']


Додатковий крок підготовки
- у категоріальних колонках є «брудні» значення (M_err_err, RED, зайві пробіли) — без очищення вони стали б окремими категоріями;
- у колонках, де значення фізично не може бути ≤ 0 (температура, світність, радіус, маса, відстань, вік, швидкість обертання, гравітація), є від'ємні значення — вони стають NaN і заповнюються разом з іншими пропусками.

In [12]:
# очищення тексту
for col in cat_cols:
    s = df[col].astype("string").str.strip()
    df[col] = s.str.replace(r"(_err)+$", "", regex=True, case=False).str.strip().str.title()
df["Spectral_Class"] = df["Spectral_Class"].str.upper()

# фізично неможливі значення перетворюються на NaN
positive_only = ["Temperature_K", "Luminosity_L_Sun", "Radius_R_Sun", "Mass_M_Sun",
                 "Distance_LY", "Age_Million_Years", "Rotation_Speed_KM_S", "Surface_Gravity"]
for col in positive_only:
    df.loc[df[col] <= 0, col] = np.nan

print("Spectral_Class:", sorted(df["Spectral_Class"].dropna().unique()))
print("Star_Color:", sorted(df["Star_Color"].dropna().unique()))

Spectral_Class: ['0', '1', '2', '3', '4', '5', '6']
Star_Color: ['0', '1', '2', '3', '4', '5', '6']


## Завдання 1.a: Заміна відсутніх значень

In [6]:
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Залишилось пропусків:", int(df.isna().sum().sum()))

Залишилось пропусків: 0


## Завдання 1.b: Кодування (Categorical Encoding) категоріальних ознак

In [7]:
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df[cat_cols].head()

,Spectral_Class,Star_Color,Constellation,Galaxy_Type,Variable_Star,Star_Type
0,4,2,8,0,0,1
1,0,4,12,0,0,3
2,4,2,12,0,0,1
3,1,1,12,0,0,6
4,5,3,6,0,0,4


## Завдання 1.c: Опрацювання аномальних значень

In [8]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower_bound, upper_bound)

## Завдання 1.d: Використання підходів масштабування ознак

In [9]:
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df.head()

,Star_ID,Temperature_K,Luminosity_L_Sun,Radius_R_Sun,Mass_M_Sun,Distance_LY,Apparent_Magnitude,Absolute_Magnitude,Spectral_Class,Star_Color,Age_Million_Years,Metallicity,Rotation_Speed_KM_S,Constellation,Galaxy_Type,Variable_Star,Surface_Gravity,Star_Type,timestamp
0,STAR_00001,-0.437781,1.918976,-0.480482,-0.069102,-0.684670,-1.230961,-1.092370,4,2,-0.875783,0.246527,-0.923321,8,0,0,0.449212,1,2025-12-23 12:40:22
1,STAR_00002,0.338968,-0.529158,-0.764435,-0.758578,1.998360,1.109626,-0.021780,0,4,-1.139117,0.166243,-0.111693,12,0,0,-0.242956,3,2025-05-16 23:10:38
2,STAR_00003,-0.435783,1.294238,0.032125,0.127990,1.998360,0.866750,-0.641667,4,2,-0.682105,-0.070596,-0.491090,12,0,0,-0.357461,1,2025-11-22 00:07:56
3,STAR_00004,-0.380580,-0.633318,-0.487317,-0.551529,-0.697691,-0.349154,1.042341,1,1,1.964608,0.900845,0.020930,12,0,0,-0.329264,6,2025-11-27 16:27:49
4,STAR_00005,-0.797563,-0.630937,-0.828433,-0.869485,-0.382485,0.787488,0.732789,5,3,0.815543,-1.499658,-1.038332,6,0,0,-0.709712,4,2025-12-01 13:38:54


## 2. Звіт про очищення даних

Під час виконання роботи було виявлено та усунуто дефекти в наборі даних astro_vision.db (10 000 записів). Для виконання завдання в датасет було додатково впроваджено синтетичні пропуски (5% у Temperature_K, Luminosity_L_Sun, Star_Color, Spectral_Class) та 50 екстремальних аномалій у Temperature_K:

- **Відсутні значення:** зафіксовано 11250 пропусків у колонках Temperature_K (1581), Luminosity_L_Sun (1057), Radius_R_Sun (575), Mass_M_Sun (575), Distance_LY (580), Apparent_Magnitude (600), Absolute_Magnitude (575), Spectral_Class (977), Star_Color (1004), Age_Million_Years (567), Metallicity (594), Rotation_Speed_KM_S (580), Constellation (441), Galaxy_Type (453), Variable_Star (481), Surface_Gravity (610). Числові пропуски відновлено за допомогою медіани, категоріальні — найчастіше вживаним значенням (модою).
- **Некоректний текст:** у категоріальних колонках усунено суфікси помилок (_err_err), зайві пробіли та різний регістр.
- **Неможливі значення:** 784 від'ємних або нульових значень у колонках, де вони фізично неможливі, замінено на NaN і відновлено медіаною.
- **Аномалії:** викиди методом IQR (1.5·IQR) приведено до допустимих меж без видалення записів (усього 10265 значень; найбільше в Luminosity_L_Sun (1834), Distance_LY (1579), Surface_Gravity (1322), Temperature_K (1170)).
- **Кодування:** категоріальні змінні перетворено на числа за допомогою Label Encoding.
- **Масштабування:** застосовано Z-score нормалізацію (StandardScaler) для числових атрибутів.

## 3. Обґрунтування методів інженерії даних

- **Заповнення пропусків:** медіана стійка до викидів (на відміну від середнього арифметичного), що критично для астрономічних даних з екстремальною світністю чи масою. Мода для тексту зберігає оригінальний розподіл класів.
- **IQR Capping:** підхід вибрано замість повного видалення рядків (Dropping), щоб не втрачати інформацію про інші коректні ознаки об'єктів. Видалення рядків призвело б до втрати даних, важливих для навчання класифікатора.
- **Label Encoding:** обрано для перетворення текстових класів у числа (0, 1, 2...). На відміну від One-Hot Encoding, цей метод не роздуває розмірність датасету (що особливо важливо для колонки Constellation, яка містить багато унікальних сузір'їв).
- **StandardScaler:** приводить ознаки з різними діапазонами (від маси ~1 до відстані ~10⁶) до єдиного масштабу.

## Завдання 4: Збереження нового датасету та створення зв'язку 1:1

У SQLite не можна додати первинний ключ до наявної таблиці командою ALTER TABLE, тому:
для оригінальної таблиці stars створюємо унікальний індекс по Star_ID (він потрібен як цільовий ключ для зовнішнього ключа),
а нову таблицю stars_data_cleaned створюємо одразу з PRIMARY KEY і FOREIGN KEY на stars.
Star_ID одночасно і первинний, і зовнішній ключ — тому кожному запису оригіналу відповідає не більше одного запису очищеної таблиці (зв'язок 1:1).

In [13]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")
...
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

#  Унікальність Star_ID в оригінальній таблиці
cur.execute('CREATE UNIQUE INDEX IF NOT EXISTS ux_stars_star_id ON stars("Star_ID");')

# Нова таблиця з PRIMARY KEY і FOREIGN KEY (зв'язок 1:1)
cur.execute("DROP TABLE IF EXISTS stars_data_cleaned;")
cols_sql = ", ".join(
    f'"{c}" REAL' if c in num_cols else
    f'"{c}" INTEGER' if c in cat_cols else
    f'"{c}" TEXT'
    for c in df.columns if c != "Star_ID"
)
cur.execute(f'''
    CREATE TABLE stars_data_cleaned (
        "Star_ID" TEXT PRIMARY KEY,
        {cols_sql},
        CONSTRAINT fk_star_id FOREIGN KEY ("Star_ID") REFERENCES stars ("Star_ID")
    );
''')
conn.commit()

# Запис очищених даних
df.to_sql("stars_data_cleaned", conn, if_exists="append", index=False)
conn.commit()

pd.read_sql_query("SELECT * FROM stars_data_cleaned", conn).head()

,Star_ID,Temperature_K,Luminosity_L_Sun,Radius_R_Sun,Mass_M_Sun,Distance_LY,Apparent_Magnitude,Absolute_Magnitude,Spectral_Class,Star_Color,Age_Million_Years,Metallicity,Rotation_Speed_KM_S,Constellation,Galaxy_Type,Variable_Star,Surface_Gravity,Star_Type,timestamp
0,STAR_00001,NaN,1.918976,NaN,NaN,NaN,-1.230961,-1.092370,4,2,NaN,0.246527,NaN,8,0,0,0.449212,1,2025-12-23 12:40:22
1,STAR_00002,0.338968,NaN,NaN,NaN,1.99836,1.109626,-0.021780,0,4,NaN,0.166243,NaN,12,0,0,NaN,3,2025-05-16 23:10:38
2,STAR_00003,NaN,1.294238,0.032125,0.12799,1.99836,0.866750,-0.641667,4,2,NaN,-0.070596,NaN,12,0,0,NaN,1,2025-11-22 00:07:56
3,STAR_00004,NaN,NaN,NaN,NaN,NaN,-0.349154,1.042341,1,1,1.964608,0.900845,0.02093,12,0,0,NaN,6,2025-11-27 16:27:49
4,STAR_00005,NaN,NaN,NaN,NaN,NaN,0.787488,0.732789,5,3,0.815543,-1.499658,NaN,6,0,0,NaN,4,2025-12-01 13:38:54


In [11]:
# Перевірка зв'язку 1:1
print("stars:", cur.execute("SELECT COUNT(*) FROM stars").fetchone()[0])
print("stars_data_cleaned:", cur.execute("SELECT COUNT(*) FROM stars_data_cleaned").fetchone()[0])
print("Записів без пари в оригіналі:", cur.execute(
    'SELECT COUNT(*) FROM stars_data_cleaned c LEFT JOIN stars s ON s."Star_ID" = c."Star_ID" WHERE s."Star_ID" IS NULL'
).fetchone()[0])
print("Порушення зовнішнього ключа:", cur.execute("PRAGMA foreign_key_check;").fetchall())
conn.close()


stars: 10000
stars_data_cleaned: 10000
Записів без пари в оригіналі: 0
Порушення зовнішнього ключа: []


- Заповнення пропусків медіаною. Медіана стійка до викидів, на відміну від середнього арифметичного. Це критично для астрономічних даних, де світність чи маса окремих зір можуть у тисячі разів перевищувати типові значення: середнє було б зміщене такими екстремумами, а медіана добре описує типовий об'єкт.
- Заповнення пропусків модою. Для текстових категорій середнє та медіана не мають змісту. Мода зберігає початковий розподіл класів і не вигадує нових категорій.
- IQR-обрізання замість видалення рядків. Метод IQR не залежить від форми розподілу так сильно, як правило трьох сигм. Обрізання (capping) замінює лише аномальну величину, зберігаючи решту коректних ознак цього об'єкта. Видалення рядків призвело б до втрати даних, важливих для навчання класифікатора, і могло порушити баланс типів зір.
- Label Encoding. Перетворює текстові класи на числа (0, 1, 2…). На відміну від One-Hot Encoding, цей метод не збільшує розмірність датасету, що важливо для колонки Constellation з великою кількістю унікальних сузір'їв. Недолік методу — він вводить штучний порядок між категоріями; для дерев рішень це не критично, а для лінійних моделей слід було б розглянути One-Hot.
- StandardScaler. Приводить ознаки з різними діапазонами до єдиного масштабу, що необхідно для моделей, чутливих до масштабу (kNN, SVM, нейромережі, логістична регресія). Для дерев рішень масштабування не обов'язкове, але й не шкодить.
- Порядок кроків. Спочатку відновлюються пропуски, бо кодувальник не може коректно обробити NaN. Кодування виконується до масштабування, щоб категорії стали числами. Викиди обробляються до масштабування, бо інакше вони спотворили б середнє та стандартне відхилення, за якими обчислюється Z-оцінка.
